<a href="https://colab.research.google.com/github/SohamManik/llm-huggingface/blob/main/LLM12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from datasets import load_dataset


raw_datasets = load_dataset("rajpurkar/squad")
raw_datasets = raw_datasets.remove_columns(["id","title"])

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [5]:
def prepare_data(example):
  answer = example["answers"]["text"][0]
  example["answer_start"] = example['answers']["answer_start"][0]
  example["answer_end"] = example["answer_start"] + len(answer)
  return example



Cleaning the data


In [6]:
raw_datasets = raw_datasets.map(prepare_data , remove_columns= ["answers"])


Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [7]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'answer_start', 'answer_end'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['context', 'question', 'answer_start', 'answer_end'],
        num_rows: 10570
    })
})

In [8]:
from huggingface_hub import notebook_login

notebook_login()

In [25]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [15]:
context = raw_datasets["train"]["context"][0]
question = raw_datasets["train"]["question"][0]

In [27]:
inputs = tokenizer(
    question,
    context,
    max_length=384,
    truncation="only_second",
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="max_length"
)

In [28]:
sample = raw_datasets["train"][0]
start_char = sample["answer_start"]
end_char = sample["answer_end"]

start_positions = []
end_positions = []

for i, offset in enumerate(inputs["offset_mapping"]):
    sequence_ids = inputs.sequence_ids(i)

    # In tokenizer(question, context), question is 0, context is 1
    idx = 0
    while idx < len(sequence_ids) and sequence_ids[idx] != 1:
        idx += 1
    context_start = idx
    while idx < len(sequence_ids) and sequence_ids[idx] == 1:
        idx += 1
    context_end = idx - 1

    # Check if answer is fully contained in this window
    if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
        start_positions.append(0)
        end_positions.append(0)
    else:
        # Find token start index
        idx = context_start
        while idx <= context_end and offset[idx][0] <= start_char:
            idx += 1
        start_positions.append(idx - 1)

        # Find token end index
        idx = context_end
        while idx >= context_start and offset[idx][1] >= end_char:
            idx -= 1
        end_positions.append(idx + 1)

print(f"Start positions: {start_positions}")
print(f"End positions: {end_positions}")

Start positions: [136]
End positions: [142]
